# Testing the CSE

In [1]:
%pip install requests pandas openpyxl tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 12.1 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Import

In [26]:
import os, time, json, hashlib, re
import requests
import pandas as pd
from pathlib import Path
import sys
sys.path.insert(0, "../src")
import config
from read_excel import load_products


## Environment variables

In [27]:
API_KEY = os.environ["GOOGLE_CSE_API_KEYS"]
CX      = os.environ["GOOGLE_CSE_CX"]

CACHE_DIR = Path("cse_cache"); CACHE_DIR.mkdir(exist_ok=True)

Creating the API call, to be reuse and to test with different filters. The API key and CX are set as environment variables, never hardcoded.

In [28]:
def cse_image_search(query: str, **filters) -> list[dict]:
    params = {
        "key": API_KEY, "cx": CX, "q": query,
        "searchType": "image", "num": 10,
        **filters,
    }
    # cache to prevent hitting the API too often, as this is testing code
    cache_id = hashlib.md5(json.dumps(
        {k: v for k, v in params.items() if k != "key"}, sort_keys=True
    ).encode()).hexdigest()
    cache_file = CACHE_DIR / f"{cache_id}.json"
    if cache_file.exists():
        return json.loads(cache_file.read_text())

    r = requests.get("https://www.googleapis.com/customsearch/v1",
                    params=params, timeout=30)
    if r.status_code == 429:
        raise RuntimeError("Daily quota exhausted (HTTP 429).")
    r.raise_for_status()
    payload = r.json()
    items = payload.get("items", [])
    total_results = payload.get("searchInformation", {}).get("totalResults")
    results = [{
        "url":          it["link"],
        "domain":       it.get("displayLink"),
        "context_url":  it.get("image", {}).get("contextLink"),
        "width":        it.get("image", {}).get("width"),
        "height":       it.get("image", {}).get("height"),
        "byte_size":    it.get("image", {}).get("byteSize"),
        "thumbnail":    it.get("image", {}).get("thumbnailLink"),
        "mime":         it.get("mime"),
        "title":        it.get("title"),
        "query":        query,
        "total_results": total_results,
    } for it in items]
    cache_file.write_text(json.dumps(results, ensure_ascii=False, indent=1))
    time.sleep(0.5)
    return results

# First test:

### Product name:
Row 140: ref='7717079985'  marca='RENAULT'  nombre='GORRA RENAULT ROJA LOGO AZUL'



In [29]:
# load_products() defaults to config.N_ROWS (10); pass a big n_rows to get the whole sheet
products = load_products(n_rows=10_000)
print(f"Loaded {len(products)} products")

# Pick a reference from the middle of the excel file as our test case
mid_idx = len(products) // 2
test_row = products.iloc[mid_idx]
ref, marca, nombre = test_row[config.COL_REF], test_row[config.COL_BRAND], test_row[config.COL_NAME]

print(f"Testing with row {mid_idx}: ref={ref!r}  marca={marca!r}  nombre={nombre!r}")

Loaded 281 products
Testing with row 140: ref='7717079985'  marca='RENAULT'  nombre='GORRA RENAULT ROJA LOGO AZUL'


In [ ]:
def make_query(marca: str, ref: str) -> str:
    """Same shape as src/query_builder.build_query: exact ref + brand."""
    return f'"{ref}" "{marca}"'

query = make_query(marca, ref)
print(query)

FILTER_SETS = {
    # --- baseline ---
    "baseline":            {"imgType": "photo", "imgSize": "large"},

    # --- test background lever (criterion 1) ---
    "white_dominant":      {"imgType": "photo", "imgSize": "large", "imgDominantColor": "white"},
    "trans_png":           {"imgType": "photo", "imgColorType": "trans", "fileType": "png"},

    # --- test resolution lever (criterion 2), one variable changed ---
    "size_xlarge":         {"imgType": "photo", "imgSize": "xlarge"},
    "size_huge":           {"imgType": "photo", "imgSize": "huge"},

    # --- test brand enforcement in isolation (criterion 5) ---
    "exact_brand":         {"imgType": "photo", "imgSize": "large", "exactTerms": marca},

    # --- test locale (criterion 5 vs 2 tradeoff) ---
    "colombia_boost":      {"imgType": "photo", "imgSize": "large", "gl": "co"},

    # --- "best guess" combos ---
    "stack_white_brand":   {"imgType": "photo", "imgSize": "large",
                            "imgDominantColor": "white", "exactTerms": marca},
    "stack_trans_brand":   {"imgType": "photo", "imgColorType": "trans",
                            "fileType": "png", "exactTerms": marca},
}

"7717079985" "RENAULT"


In [31]:
results = []
for name, filters in FILTER_SETS.items():
    items = cse_image_search(query, **filters)
    for it in items:
        it["variant"] = name
        it["ref"] = ref
    results.extend(items)
    print(f"{name:>18}: {len(items)} results")

pool = pd.DataFrame(results)
pool.head()

          baseline: 10 results
    white_dominant: 8 results
         trans_png: 0 results
       size_xlarge: 0 results
         size_huge: 0 results
       exact_brand: 10 results
    colombia_boost: 10 results
 stack_white_brand: 8 results
 stack_trans_brand: 0 results


,url,domain,context_url,width,height,byte_size,thumbnail,mime,title,query,total_results,variant,ref
0,https://contactos.renault.com.co/the-originals...,contactos.renault.com.co,https://contactos.renault.com.co/the-originals...,465,467,114622,https://encrypted-tbn0.gstatic.com/images?q=tb...,image/jpeg,Catálogo The Originals | Renault Co,"""7717079985"" ""RENAULT""",26,baseline,7717079985
1,https://contactos.renault.com.co/the-originals...,contactos.renault.com.co,https://contactos.renault.com.co/the-originals...,465,467,57612,https://encrypted-tbn0.gstatic.com/images?q=tb...,image/jpeg,Catálogo The Originals | Renault Co,"""7717079985"" ""RENAULT""",26,baseline,7717079985
2,https://contactos.renault.com.co/the-originals...,contactos.renault.com.co,https://contactos.renault.com.co/the-originals...,465,467,62755,https://encrypted-tbn0.gstatic.com/images?q=tb...,image/jpeg,Catálogo The Originals | Renault Co,"""7717079985"" ""RENAULT""",26,baseline,7717079985
3,https://contactos.renault.com.co/the-originals...,contactos.renault.com.co,https://contactos.renault.com.co/the-originals...,465,467,67036,https://encrypted-tbn0.gstatic.com/images?q=tb...,image/jpeg,Catálogo The Originals | Renault Co,"""7717079985"" ""RENAULT""",26,baseline,7717079985
4,https://contactos.renault.com.co/the-originals...,contactos.renault.com.co,https://contactos.renault.com.co/the-originals...,465,467,58886,https://encrypted-tbn0.gstatic.com/images?q=tb...,image/jpeg,Catálogo The Originals | Renault Co,"""7717079985"" ""RENAULT""",26,baseline,7717079985


In [32]:
summary = (
    pool.groupby("variant")
    .agg(n_returned=("url", "size"), total_results=("total_results", "first"))
    .reindex(FILTER_SETS.keys())
)
summary["n_returned"] = summary["n_returned"].fillna(0).astype(int)
summary["total_results"] = pd.to_numeric(summary["total_results"], errors="coerce")
summary


,n_returned,total_results
variant,,
baseline,10,26.0
white_dominant,8,8.0
trans_png,0,NaN
size_xlarge,0,NaN
size_huge,0,NaN
exact_brand,10,26.0
colombia_boost,10,115.0
stack_white_brand,8,8.0
stack_trans_brand,0,NaN


In [33]:
from IPython.display import HTML

def show_pool(ref, variant=None, n=10):
    """Show thumbnails for a ref, grouped by filter variant so they're easy to compare."""
    sub = pool[pool.ref == ref]
    if variant:
        sub = sub[sub.variant == variant]
    blocks = []
    for name, group in sub.groupby("variant"):
        cells = "".join(
            f'<div style="display:inline-block;margin:4px;text-align:center;width:160px">'
            f'<img src="{r.thumbnail}" style="max-width:150px;max-height:150px">'
            f'<div style="font-size:10px">{r.domain}<br>{r.width}x{r.height}</div></div>'
            for r in group.head(n).itertuples())
        blocks.append(f"<h4>{name}</h4><div>{cells}</div>")
    return HTML("".join(blocks))

show_pool(ref)

## Testing without literal search 

In [37]:
def make_query_non_literal(marca: str, ref: str) -> str:
    """Loose keyword query: no exact-phrase quoting, lets Google match non-adjacent/reordered terms."""
    return f'{ref} {marca}'

In [38]:
results = []
for name, filters in FILTER_SETS.items():
    items = cse_image_search(query, **filters)
    for it in items:
        it["variant"] = name
        it["ref"] = ref
    results.extend(items)
    print(f"{name:>18}: {len(items)} results")

pool = pd.DataFrame(results)
pool.head()

          baseline: 10 results
    white_dominant: 8 results
         trans_png: 0 results
       size_xlarge: 0 results
         size_huge: 0 results
       exact_brand: 10 results
    colombia_boost: 10 results
 stack_white_brand: 8 results
 stack_trans_brand: 0 results


,url,domain,context_url,width,height,byte_size,thumbnail,mime,title,query,total_results,variant,ref
0,https://contactos.renault.com.co/the-originals...,contactos.renault.com.co,https://contactos.renault.com.co/the-originals...,465,467,114622,https://encrypted-tbn0.gstatic.com/images?q=tb...,image/jpeg,Catálogo The Originals | Renault Co,"""7717079985"" ""RENAULT""",26,baseline,7717079985
1,https://contactos.renault.com.co/the-originals...,contactos.renault.com.co,https://contactos.renault.com.co/the-originals...,465,467,57612,https://encrypted-tbn0.gstatic.com/images?q=tb...,image/jpeg,Catálogo The Originals | Renault Co,"""7717079985"" ""RENAULT""",26,baseline,7717079985
2,https://contactos.renault.com.co/the-originals...,contactos.renault.com.co,https://contactos.renault.com.co/the-originals...,465,467,62755,https://encrypted-tbn0.gstatic.com/images?q=tb...,image/jpeg,Catálogo The Originals | Renault Co,"""7717079985"" ""RENAULT""",26,baseline,7717079985
3,https://contactos.renault.com.co/the-originals...,contactos.renault.com.co,https://contactos.renault.com.co/the-originals...,465,467,67036,https://encrypted-tbn0.gstatic.com/images?q=tb...,image/jpeg,Catálogo The Originals | Renault Co,"""7717079985"" ""RENAULT""",26,baseline,7717079985
4,https://contactos.renault.com.co/the-originals...,contactos.renault.com.co,https://contactos.renault.com.co/the-originals...,465,467,58886,https://encrypted-tbn0.gstatic.com/images?q=tb...,image/jpeg,Catálogo The Originals | Renault Co,"""7717079985"" ""RENAULT""",26,baseline,7717079985


In [39]:
summary = (
    pool.groupby("variant")
    .agg(n_returned=("url", "size"), total_results=("total_results", "first"))
    .reindex(FILTER_SETS.keys())
)
summary["n_returned"] = summary["n_returned"].fillna(0).astype(int)
summary["total_results"] = pd.to_numeric(summary["total_results"], errors="coerce")
summary


,n_returned,total_results
variant,,
baseline,10,26.0
white_dominant,8,8.0
trans_png,0,NaN
size_xlarge,0,NaN
size_huge,0,NaN
exact_brand,10,26.0
colombia_boost,10,115.0
stack_white_brand,8,8.0
stack_trans_brand,0,NaN


# Gemini API Calls